## BumbleBee

Decoder only Transformer build from scratch

In [1]:
# Using Tiny Stories Dataset. (I Know you'll say this is validation version of the dataset, but bear with me, I don't have a powerful GPU)
!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt

--2026-07-07 08:19:45--  https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt
Resolving huggingface.co (huggingface.co)... 3.167.112.38, 3.167.112.96, 3.167.112.25, ...
Connecting to huggingface.co (huggingface.co)|3.167.112.38|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/645e8da96320b0efe40ade7a/a41b122d3e83c4f0e375e4f6df4ebe55a9a4692c9b6043ca6356d740e75cd312?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27TinyStories-valid.txt%3B+filename%3D%22TinyStories-valid.txt%22%3B&response-content-type=text%2Fplain&user_id=public&Expires=1783415985&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjQ1ZThkYTk2MzIwYjBlZmU0MGFkZTdhL2E0MWIxMjJkM2U4M2M0ZjBlMzc1ZTRmNmRmNGViZTU1YTlhNDY5MmM5YjYwNDNjYTYzNTZkNzQwZTc1Y2QzMTJcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitmaWxlbmFtZ

In [2]:
# read it in to inspect it
with open('TinyStories-valid.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset in characters: ", len(text)) # a respectable size right.

length of dataset in characters:  19432979


In [4]:
# let's look at the first 1000 characters
print(text[:1000])

 Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."
After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.
<|endoftext|>
Once upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy loved to climb. She climbed trees, rocks, and hills. One day, Roxy found an icy hill. She had never seen anything like it before. It was shiny and cold, and she wanted to climb it.
Roxy tried to climb the icy hill, but it was very slippery. She tried again and again, but she kept falling down. Roxy was sad. She wanted to climb the icy hill so much. Then, she saw a little bird named Billy. Billy saw that Roxy was sad and asked, "Why are you sad, Roxy?"
Roxy told Billy about the icy hill and how she couldn't climb it. Billy said, "I have an idea! Let's fi

In [5]:
# here are all the unique characters that occur in this text. Since this is a character level model, we'll have these as our unique tokens.
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"#$&'()*+,-./0123456789:;<>?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz|­´éñ ​–—‘’“”…🎓
98


In [6]:
# create a mapping from characters to integers So this will be the backbone of our encoder and decoder
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: This will take in a string and output the tokens
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of tokens, output a string

print(encode("Hi BumbleBee Here"))
print(decode(encode("Hi BumbleBee Here")))

[38, 65, 1, 32, 77, 69, 58, 68, 61, 32, 61, 61, 1, 38, 61, 74, 61]
Hi BumbleBee Here


In [8]:
# let's now encode the entire dataset and store it into a Tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the first 1000 characters will look like this. [This tiny dataset took a whooping 18s to be transformed. Imagine using the 1.9 gb one. hehe]

torch.Size([19432979]) torch.int64
tensor([ 1, 49, 72, 71, 76, 14,  1, 49, 72, 71, 76,  1, 75, 57, 79,  1, 76, 64,
        61,  1, 75, 64, 65, 70, 81,  1, 59, 57, 74,  1, 57, 70, 60,  1, 75, 57,
        65, 60, 12,  1,  3, 53, 71, 79, 12,  1, 41, 65, 76, 76, 81, 12,  1, 81,
        71, 77, 74,  1, 59, 57, 74,  1, 65, 75,  1, 75, 71,  1, 58, 74, 65, 63,
        64, 76,  1, 57, 70, 60,  1, 59, 68, 61, 57, 70,  2,  3,  1, 41, 65, 76,
        76, 81,  1, 75, 69, 65, 68, 61, 60,  1, 57, 70, 60,  1, 74, 61, 72, 68,
        65, 61, 60, 12,  1,  3, 50, 64, 57, 70, 67,  1, 81, 71, 77, 12,  1, 49,
        72, 71, 76, 14,  1, 39,  1, 72, 71, 68, 65, 75, 64,  1, 65, 76,  1, 61,
        78, 61, 74, 81,  1, 60, 57, 81, 14,  3,  0, 31, 62, 76, 61, 74,  1, 72,
        68, 57, 81, 65, 70, 63,  1, 79, 65, 76, 64,  1, 76, 64, 61,  1, 59, 57,
        74, 12,  1, 41, 65, 76, 76, 81,  1, 57, 70, 60,  1, 49, 72, 71, 76,  1,
        62, 61, 68, 76,  1, 76, 64, 65, 74, 75, 76, 81, 14,  1, 50, 64, 61, 81,
     

In [9]:
# Let's now split up the data into train and validation sets so that we can factor out the possibility of overfitting.
n = int(0.9*len(data)) # 90% training and the rest validation dataset.
train_data = data[:n]
val_data = data[n:]

In [10]:
block_size = 8 # block size is the max-context length our transformer will see at a time.
train_data[:block_size+1] # thus if it has 8 letters it will predict the 9th letter

tensor([ 1, 49, 72, 71, 76, 14,  1, 49, 72])

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([1]) the target: 49
when input is tensor([ 1, 49]) the target: 72
when input is tensor([ 1, 49, 72]) the target: 71
when input is tensor([ 1, 49, 72, 71]) the target: 76
when input is tensor([ 1, 49, 72, 71, 76]) the target: 14
when input is tensor([ 1, 49, 72, 71, 76, 14]) the target: 1
when input is tensor([ 1, 49, 72, 71, 76, 14,  1]) the target: 49
when input is tensor([ 1, 49, 72, 71, 76, 14,  1, 49]) the target: 72


In [12]:
torch.manual_seed(127) # bumblebee's code was B-127 hehe... :) Easter Egg
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[61,  1, 79, 57, 75,  1, 75, 71],
        [57, 70, 60,  1, 76, 64, 61, 81],
        [68, 65, 76, 76, 68, 61,  1, 60],
        [63,  1, 63, 71, 57, 68,  1, 71]])
targets:
torch.Size([4, 8])
tensor([[ 1, 79, 57, 75,  1, 75, 71,  1],
        [70, 60,  1, 76, 64, 61, 81,  1],
        [65, 76, 76, 68, 61,  1, 60, 77],
        [ 1, 63, 71, 57, 68,  1, 71, 70]])
----
when input is [61] the target: 1
when input is [61, 1] the target: 79
when input is [61, 1, 79] the target: 57
when input is [61, 1, 79, 57] the target: 75
when input is [61, 1, 79, 57, 75] the target: 1
when input is [61, 1, 79, 57, 75, 1] the target: 75
when input is [61, 1, 79, 57, 75, 1, 75] the target: 71
when input is [61, 1, 79, 57, 75, 1, 75, 71] the target: 1
when input is [57] the target: 70
when input is [57, 70] the target: 60
when input is [57, 70, 60] the target: 1
when input is [57, 70, 60, 1] the target: 76
when input is [57, 70, 60, 1, 76] the target: 64
when input is [57, 70, 6

In [ ]:
print(xb) # our input to the transformer

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(127)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
# our loss should've been -log(1/vocab_size) ie -log(1/98) = 2 at initialization
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


In [ ]:
# create a PyTorch optimizer || We could've used SGD but it would've been slower. I don't know the workings of this model but I know it's faster than SGD
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(10000): # optimizing 10000 times

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


In [ ]:
# Ran the optimization phase twice. Got final loss as 2.12. Much better results if you seee. Still Gibberish tho.
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))

## The mathematical trick in self-attention

In [ ]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

In [ ]:
# version 1: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)

In [ ]:
# version 2: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow2, xbow3)


In [ ]:
# version 3: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

In [ ]:
wei[0]

Notes I made along the way while learning to make this:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [ ]:
k.var()

In [ ]:
q.var()

In [ ]:
wei.var()

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

In [3]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

NameError: name 'torch' is not defined

In [ ]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

In [ ]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

In [ ]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

### Full finished code. Final implementation:


In [13]:
# all the default imports
import torch
import torch.nn as nn
from torch.nn import functional as F

In [27]:
# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 128 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 8
n_layer = 8
dropout = 0.2
torch.manual_seed(127)

In [28]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


In [29]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [30]:
# definition for a single head of attention
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

In [31]:
# just grouping multiple single heads and concatenating
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [32]:
# simple feed forward layer
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [33]:
# sandwiching attention heads and feedforward layer as one block
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [34]:
class BumbleBee(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [35]:
# init. [Please don't run this code block. Running it will reset the model]
model = BumbleBee()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

0.419298 M parameters


In [36]:
# Training phase
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.8018, val loss 4.8006
step 100: train loss 2.4752, val loss 2.4637
step 200: train loss 2.3404, val loss 2.3360
step 300: train loss 2.2711, val loss 2.2697
step 400: train loss 2.1923, val loss 2.1960
step 500: train loss 2.1128, val loss 2.1107
step 600: train loss 2.0001, val loss 2.0066
step 700: train loss 1.9022, val loss 1.9087
step 800: train loss 1.8111, val loss 1.8285
step 900: train loss 1.7340, val loss 1.7514
step 1000: train loss 1.6797, val loss 1.6977
step 1100: train loss 1.6372, val loss 1.6579
step 1200: train loss 1.5958, val loss 1.6105
step 1300: train loss 1.5581, val loss 1.5808
step 1400: train loss 1.5195, val loss 1.5405
step 1500: train loss 1.4954, val loss 1.5097
step 1600: train loss 1.4723, val loss 1.4862
step 1700: train loss 1.4530, val loss 1.4744
step 1800: train loss 1.4326, val loss 1.4530
step 1900: train loss 1.4101, val loss 1.4365
step 2000: train loss 1.3899, val loss 1.4084
step 2100: train loss 1.3753, val loss 1.3980


In [26]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


<|endoftext|>

The day was walking picked, Macher was very bad. He made his talk her peerful and lived adigato.
The flow pulled all of the that pretty tiving, flover aling a mine aunturant for the frog. They him felt for the catinum, didn's my late one.
Lily said it was their seal, it was a big slide. She saw his mommy said, "We shiny, and life, just always tulb others the bake cat for cold. Clum ye like under it fun." 
Mia heard her as have a boy offlet pan and appeared the tay fountaininagting. He enver the giants time together getting on his it was a. The girtcher was bright along on the tois.
One day, Timmy's mom dad gad-old so more food much see and hight. From the garden all that was the share ahole and hugged hice and waighed felt.
"Good, you always eldied a part toy some mecratime coins." 
Tim didn't finish anymored to shims and have fun it from the wisher had done lives Mom. Tom feed a lick in the sunzing with their little girl. She gave knew that she his mom highey were each